# 🧪 [Colab 실습] PyTorch 양자화 미니랩 — PTQ 7단계를 진짜 모델로

**온디바이스 AI 프로그래밍 · 「가상 NPU 실습」 다음 단계 — Day 2 본 실습(XWN·QAT) 직전 브릿지**

| 항목 | 내용 |
| --- | --- |
| 실습 목표 | 가상 NPU 실습에서 numpy로 흉내 낸 양자화를 **실제 PyTorch 프레임워크**로 재현 |
| 환경 | Google Colab — **GPU 런타임 권장**(학습 2~3분), CPU도 가능(학습 축소 모드 자동 적용) |
| 데이터/모델 | CIFAR-10 + 직접 만드는 소형 CNN (양자화 친화 구조) |
| 핵심 API | `torch.ao.quantization` — fuse / prepare / calibrate / convert / prepare_qat |

## 이 실습의 위치 — 세 실습의 관계

```text
[완료] 가상 NPU 실습        : quantize()를 numpy로 직접 구현 — "원리"
[지금] PyTorch 양자화 미니랩 : 같은 일을 프레임워크 API로 — "도구"
[다음] Day 2 본 실습        : 같은 일을 디퍼아이 DDesignerAPI/컴파일러로 — "실전"
```

## 실습 로드맵 (교안 Day 2 「PTQ 실전 7단계」와 1:1 대응)

| Part | 주제 | 교안 PTQ 7단계 |
| --- | --- | --- |
| 1 | 데이터·모델 준비와 FP32 기준선 | — |
| 2 | ★ PTQ 7단계 그대로 따라하기 | ①~⑦ 전체 |
| 3 | 캘리브레이션 대표성 실험 (가상실습 Part 6-4의 실전판) | ④ 심화 |
| 4 | Per-tensor vs Per-channel 대결 | ② 심화 |
| 5 | INT8 vs FP32 추론 속도 실측 | ⑥ 심화 |
| 6 | (선택) QAT 맛보기 — PTQ→QAT 결정 규칙 체험 | 교안 QAT 전략 |
| 7 | 리포트 과제 & Day 2 연결 | — |

> 💡 각 Step의 **✅ 확인**을 점검하고 **✏️ 직접 해보기**로 실험을 확장하세요.


---
# Part 0. 환경 준비

### Step 0-1. 런타임 확인

**런타임 → 런타임 유형 변경 → T4 GPU** 를 권장합니다 (학습이 2~3분으로 단축).
CPU만 있어도 실습은 완주 가능하며, 아래 셀이 자동으로 축소 모드를 켭니다.

> ⚠️ **중요한 사전 지식:** 학습은 GPU에서 해도 되지만, **PyTorch 양자화 모델(INT8)의 실행은 CPU 전용**입니다
> (fbgemm/qnnpack 백엔드가 CPU용이기 때문). 그래서 이 실습은 "GPU로 학습 → CPU로 양자화·추론" 흐름을 따릅니다.
> 이는 실물에서 "GPU/Colab으로 학습 → NPU로 추론"하는 Day 2 흐름의 축소판이기도 합니다.

In [ ]:
import copy, io, time, random
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.ao.quantization as tq
import matplotlib.pyplot as plt

# 한글 폰트 (실패해도 실습 무관)
try:
    import subprocess, matplotlib.font_manager as fm
    subprocess.run(["apt-get", "install", "-y", "fonts-nanum"], capture_output=True, timeout=120)
    fm.fontManager.addfont("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    plt.rc("font", family="NanumGothic")
except Exception:
    pass
plt.rc("axes", unicode_minus=False)

torch.manual_seed(42); random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
torch.backends.quantized.engine = "fbgemm"     # x86 CPU용 INT8 백엔드 (교안: ARM이면 qnnpack)

print("torch      :", torch.__version__)
print("학습 장치  :", device, "(GPU 없음 → 축소 모드)" if device == "cpu" else "")
print("양자화 엔진:", torch.backends.quantized.engine, "| 지원:", torch.backends.quantized.supported_engines)

---
# Part 1. 데이터·모델 준비와 FP32 기준선

### Step 1-1. CIFAR-10 로드

32×32 컬러 이미지 10클래스. 다운로드 포함 약 1분 소요됩니다.
CPU 모드에서는 학습 데이터를 1만 장으로 줄여 시간을 아낍니다.

In [ ]:
# 1. data 디렉토리 생성
!mkdir -p ./data

# 2. fast.ai S3 CDN에서 고속 다운로드 및 압축 해제
!wget -q --show-progress -O ./data/cifar10.tgz https://s3.amazonaws.com/fast-ai-imageclas/cifar10.tgz
!tar -xzf ./data/cifar10.tgz -C ./data/

# 3. torchvision 구조에 맞게 폴더 이름 변경 및 정리
!mv ./data/cifar10 ./data/cifar-10-batches-py
!rm ./data/cifar10.tgz

!echo "다운로드 완료!"

In [ ]:
import torch
import torchvision
import torchvision.transforms as T

# 1. 전처리 설정
tf = T.Compose([
    T.ToTensor(),
    T.Normalize((0.49, 0.48, 0.45), (0.25, 0.24, 0.26))
])

# 2. fast.ai 폴더 구조(train, test)에 맞춰 ImageFolder로 로드
# fast.ai 데이터셋은 ./data/cifar10/train 과 ./data/cifar10/test 구조를 가집니다.
train_full = torchvision.datasets.ImageFolder("./data/cifar-10-batches-py/train", transform=tf)
test_set   = torchvision.datasets.ImageFolder("./data/cifar-10-batches-py/test", transform=tf)

# 3. CPU/GPU 여부에 따른 학습셋 크기 조절
device = "cuda" if torch.cuda.is_available() else "cpu"
n_train = len(train_full) if device == "cuda" else 10_000
train_set = torch.utils.data.Subset(train_full, range(n_train))

# 4. DataLoader 생성
train_loader = torch.utils.data.DataLoader(train_set, batch_size=128, shuffle=True, num_workers=2)
test_loader  = torch.utils.data.DataLoader(test_set,  batch_size=256, shuffle=False, num_workers=2)

classes = train_full.classes
print(f"학습 {len(train_set):,}장 / 평가 {len(test_set):,}장 준비 완료! (클래스 수: {len(classes)})")

### Step 1-2. 양자화 친화 모델 정의 — 구조에 담긴 3가지 장치

아무 모델이나 양자화되는 게 아닙니다. 다음 세 가지가 **양자화를 위한 설계**입니다.

| 장치 | 역할 | 가상 NPU 실습과의 연결 |
| --- | --- | --- |
| `QuantStub` / `DeQuantStub` | FP32↔INT8 경계 표시 — "여기부터 정수 세계" | `quantize()` / 재양자화 후 복원 지점 |
| Conv→BN→ReLU **순서 고정** | 셋을 하나로 융합(fuse)하기 위한 배치 | Part 10에서 언급한 fusion |
| `ReLU(inplace=False)` | fuse 시 모듈 교체가 안전하도록 | — |

In [ ]:
class ConvBNReLU(nn.Sequential):
    """융합(fuse) 가능한 표준 블록 — 이름 없는 Sequential이라 인덱스 '0','1','2'로 접근"""
    def __init__(self, cin, cout, stride=1):
        super().__init__(
            nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(cout),
            nn.ReLU(inplace=False),
        )

class MiniCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.quant   = tq.QuantStub()      # ← FP32 입력이 INT8로 들어가는 문
        self.block1  = ConvBNReLU(3,   32)
        self.block2  = ConvBNReLU(32,  64, stride=2)
        self.block3  = ConvBNReLU(64, 128, stride=2)
        self.block4  = ConvBNReLU(128,128)
        self.pool    = nn.AdaptiveAvgPool2d(1)
        self.fc      = nn.Linear(128, num_classes)
        self.dequant = tq.DeQuantStub()    # ← INT8 결과가 FP32로 나오는 문

    def forward(self, x):
        x = self.quant(x)
        x = self.block4(self.block3(self.block2(self.block1(x))))
        x = self.pool(x).flatten(1)
        x = self.fc(x)
        return self.dequant(x)

    def fuse_model(self):
        """각 블록의 Conv+BN+ReLU를 하나의 모듈로 융합 (반드시 eval 모드에서!)"""
        for name in ["block1", "block2", "block3", "block4"]:
            tq.fuse_modules(getattr(self, name), [["0", "1", "2"]], inplace=True)

model = MiniCNN().to(device)
n_params = sum(p.numel() for p in model.parameters())
print(model.block1)
print(f"\n파라미터 {n_params:,}개 — BlackSwan 모델 제한(≈30MB)에 한참 여유인 교육용 소형 모델")

### Step 1-3. 짧은 학습 — 양자화 실험의 재료 만들기

목적은 SOTA가 아니라 **"양자화 전후를 비교할 수 있을 만큼 학습된 모델"**입니다.
GPU 3 epoch ≈ 2~3분, CPU 1 epoch ≈ 5~8분. (실행 후 잠시 커피 타임 ☕)

In [ ]:
def evaluate(m, loader, max_batches=None, dev="cpu"):
    m.eval(); correct = total = 0
    with torch.no_grad():
        for bi, (x, y) in enumerate(loader):
            if max_batches and bi >= max_batches: break
            out = m(x.to(dev))
            correct += (out.argmax(1).cpu() == y).sum().item()
            total   += y.numel()
    return correct / total * 100

EPOCHS = 3 if device == "cuda" else 1
opt  = torch.optim.SGD(model.parameters(), lr=0.05, momentum=0.9, weight_decay=5e-4)
sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(train_loader))
crit = nn.CrossEntropyLoss()

model.train()
for ep in range(EPOCHS):
    t0, run = time.time(), 0.0
    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        opt.zero_grad()
        loss = crit(model(x), y)
        loss.backward(); opt.step(); sch.step()
        run += loss.item()
    print(f"epoch {ep+1}/{EPOCHS}  loss {run/len(train_loader):.3f}  ({time.time()-t0:.0f}s)")

model = model.cpu()          # ★ 이후 모든 양자화 작업은 CPU에서 (fbgemm은 CPU 전용)
print("학습 완료 — 모델을 CPU로 이동")

### Step 1-4. FP32 기준선 측정 — 정확도·크기·속도의 "정답지"

Day 1 리포트에서 NPU 성능을 기록했듯, 여기서도 3축 기준선을 먼저 확보합니다.
이후 모든 양자화 결과는 이 숫자와 비교됩니다.

In [ ]:
def model_size_kb(m):
    buf = io.BytesIO(); torch.save(m.state_dict(), buf)
    return buf.getbuffer().nbytes / 1024

def bench_ms(m, batch=16, n=30):
    xb = torch.randn(batch, 3, 32, 32)
    m.eval()
    with torch.no_grad():
        for _ in range(5): m(xb)                       # warmup (Day 5 벤치마크 표준)
        t0 = time.perf_counter()
        for _ in range(n): m(xb)
    return (time.perf_counter() - t0) / n * 1000

fp32_acc  = evaluate(model, test_loader)
fp32_size = model_size_kb(model)
fp32_ms   = bench_ms(model)

print("═══ FP32 기준선 ═══")
print(f"정확도(테스트 1만장): {fp32_acc:.2f}%")
print(f"모델 크기           : {fp32_size:.0f} KB")
print(f"CPU 추론(batch 16)  : {fp32_ms:.1f} ms")

> **✅ Part 1 확인**
> - [ ] QuantStub/DeQuantStub의 역할(정수 세계의 입구/출구)을 설명할 수 있다
> - [ ] Conv-BN-ReLU 순서가 fuse를 위한 설계임을 이해했다
> - [ ] FP32 3축 기준선(정확도·크기·속도)을 확보했다

---
# Part 2. ★ PTQ 7단계 그대로 따라하기

교안 Day 2 「PTQ 실전 7단계」를 한 단계 = 한 셀로 실행합니다.

```text
① 사전학습 모델 준비 → ② 양자화 Config 선택 → ③ Observer 삽입
→ ④ Calibration 수행 → ⑤ Convert 실행 → ⑥ 정확도 검증 → ⑦ ONNX Export
```

### Step 2-1. [①] 사전학습 모델 준비 — `.eval()` 모드 필수

교안: "*.pth 체크포인트 로드, .eval() 모드 필수*".
왜 필수일까요? BatchNorm이 train 모드면 배치 통계를 계속 갱신하고 Dropout이 켜져 있어
**캘리브레이션 때 관찰하는 값 분포가 배포 시와 달라지기 때문**입니다.

In [ ]:
ptq_model = copy.deepcopy(model)     # 원본 보존 (이후 실험들이 각자 복사해서 사용)
ptq_model.eval()                      # ★ 필수
print("모드 확인 — training:", ptq_model.training, "(False여야 함)")

### Step 2-2. [② 전처리] Fuse — Conv+BN+ReLU를 한 덩어리로

qconfig를 붙이기 전에 융합을 먼저 합니다. 두 가지 이유입니다.

1. **정확도**: BN을 Conv의 weight/bias에 수학적으로 접어 넣으면(folding) 양자화할 연산이 줄어 오차 누적이 감소
2. **속도**: 가상 NPU 실습 Part 10에서 본 "메모리 왕복 제거" — 중간 결과를 쓰고 읽는 비용이 사라짐

실행 후 블록 구조가 어떻게 바뀌는지 직접 확인하세요.

In [ ]:
print("── fuse 전 block1 ──")
print(ptq_model.block1)

ptq_model.fuse_model()

print("\n── fuse 후 block1 ──")
print(ptq_model.block1)
print("\n💡 Conv2d+BN+ReLU 세 모듈 → ConvReLU2d 하나 + Identity 둘")
print("   BN이 사라진 게 아니라 Conv의 weight/bias 안으로 '접혀 들어간' 것입니다 (BN folding).")

acc_fused = evaluate(ptq_model, test_loader, max_batches=8)
print(f"\nfuse 후 정확도(표본): {acc_fused:.2f}% — FP32 {fp32_acc:.2f}%와 사실상 동일 (수학적 등가 변환)")

### Step 2-3. [②] 양자화 Config 선택 — fbgemm

교안: "*fbgemm(x86) / qnnpack(ARM), per-channel 권장*".
`get_default_qconfig("fbgemm")`는 **weight는 per-channel, activation은 per-tensor**로 설정된
표준 조합입니다 (왜 이 조합인지는 Part 4에서 실험으로 확인).

In [ ]:
ptq_model.qconfig = tq.get_default_qconfig("fbgemm")
print(ptq_model.qconfig)
print("\nweight     → per_channel_symmetric (채널별 scale)")
print("activation → per_tensor_affine + HistogramObserver (분포 관찰 후 범위 결정)")

### Step 2-4. [③] Observer 삽입 — `prepare()`

각 레이어에 **통계 수집기(Observer)**가 자동으로 붙습니다.
가상 NPU 실습의 `quantize()`에서 `np.abs(x).max()`로 r_max를 구했던 것의
프레임워크 버전이 바로 이 Observer입니다.

In [ ]:
tq.prepare(ptq_model, inplace=True)
print(ptq_model.block1)
print("\n💡 ConvReLU2d 뒤에 activation_post_process(HistogramObserver)가 붙었습니다.")
print("   아직 모델은 FP32 — Observer는 '구경만' 하며 값 분포를 기록할 준비 중입니다.")

### Step 2-5. [④] Calibration — 대표 데이터 순전파

교안: "*대표 데이터 300~500장, 순전파만 — 기울기 X*".
학습 데이터에서 300장을 흘려보내면 Observer들이 레이어별 활성값 분포(r_min, r_max)를 기록합니다.
**이 순간이 scale과 zero-point가 결정되는 순간**입니다.

In [ ]:
CALIB_N = 300
calib_imgs = torch.stack([train_set[i][0] for i in range(CALIB_N)])

ptq_model.eval()
with torch.no_grad():                          # 기울기 X — 순전파만
    for i in range(0, CALIB_N, 64):
        ptq_model(calib_imgs[i:i+64])

obs = ptq_model.block1[0].activation_post_process
print(f"캘리브레이션 완료 — {CALIB_N}장")
print(f"block1 activation 관찰 범위: [{obs.min_val:.3f}, {obs.max_val:.3f}]")
print("→ 이 범위가 INT8 [0,255](quint8)로 매핑될 scale을 결정합니다 (가상실습 scale 공식과 동일 원리)")

### Step 2-6. [⑤] Convert — INT8로 고정

Observer가 모은 통계로 scale/zero-point를 계산하고, weight를 실제 INT8로 변환하며,
연산 모듈을 양자화 커널로 교체합니다. **컴파일 시점에 상수가 박제되는 순간**
(가상 NPU 실습 6장, Day 2 컴파일 ④단계)의 프레임워크 버전입니다.

In [ ]:
tq.convert(ptq_model, inplace=True)
print(ptq_model.block1)
print("\n💡 QuantizedConvReLU2d — scale과 zero_point가 모듈 안에 상수로 박제되었습니다.")

### Step 2-7. [⑥] 정확도 검증 + 크기 비교

교안 목표: "*FP32 대비 손실 2%p 이내, 모델 크기 75% 감소*".

In [ ]:
ptq_acc  = evaluate(ptq_model, test_loader)
ptq_size = model_size_kb(ptq_model)

print(f"{'':12}{'FP32':>10}{'INT8(PTQ)':>12}{'변화':>14}")
print(f"{'정확도':10}{fp32_acc:>9.2f}%{ptq_acc:>11.2f}%{ptq_acc-fp32_acc:>+12.2f}%p")
print(f"{'크기':11}{fp32_size:>8.0f}KB{ptq_size:>10.0f}KB{'':>6}{fp32_size/ptq_size:.1f}배 감소")
print()
verdict = "✅ PTQ로 충분 (교안 결정 규칙: 손실 ≤ 2%p)" if fp32_acc - ptq_acc <= 2.0 \
          else "⚠️ 손실 > 2%p → 교안 결정 규칙에 따라 QAT로 전환 (Part 6에서 실습!)"
print(verdict)

### Step 2-8. INT8 weight 해부 — 가상 NPU 실습과의 만남 🤝

양자화된 Conv 안에는 무엇이 들어 있을까요? 실제 int8 정수와 per-channel scale을 꺼내 보고,
가상 실습에서 배운 복원 공식 `float ≈ int8 × scale`이 성립하는지 직접 검증합니다.

In [ ]:
qconv = ptq_model.block1[0]
w = qconv.weight()                              # 양자화된 weight 텐서

print("dtype        :", w.dtype)
print("int8 원시값  :", w.int_repr()[0, 0, 0].tolist(), "(첫 채널·첫 행)")
scales = w.q_per_channel_scales()
print("채널별 scale :", scales.shape, "— 출력 채널 32개 각각의 scale")
print("scale 예시   :", [f"{s:.5f}" for s in scales[:4].tolist()])

# 복원 검증: int8 x scale ≈ 원본 FP32(BN folding된) weight
ref = copy.deepcopy(model).eval(); ref.fuse_model()      # fuse된 FP32 정답지
w_fp   = ref.block1[0][0].weight.detach()   # ConvReLU2d 내부의 Conv2d
w_deq  = w.dequantize()
max_err = (w_deq - w_fp).abs().max().item()
print(f"\n복원 오차 max|int8xscale - fp32| = {max_err:.6f}")
print(f"scale의 절반({scales.max().item()/2:.6f}) 이하 → 반올림 오차뿐, 복원 공식 성립 ✅")
print("\n💡 가상 실습 Part 6의 quantize()와 완전히 같은 수학 — 도구만 numpy → PyTorch로 바뀌었습니다.")

### Step 2-9. [⑦] ONNX Export — 여기서부터가 Day 2 본 실습

교안 7단계의 마지막은 "*opset 13+ · 고정 shape → NPU 컴파일 준비 완료*"입니다.
이 노트북에서는 개념 확인만 하고, 실제 Export→컴파일→`.tachyrt`는 Day 2 본 실습에서
디퍼아이 툴체인으로 수행합니다. (2순위 추천 실습 「ONNX 그래프 해부」에서 별도 심화 가능)

> **✅ Part 2 확인**
> - [ ] PTQ 7단계 각각이 코드 어느 줄인지 짚을 수 있다
> - [ ] fuse가 BN folding + 메모리 왕복 제거임을 설명할 수 있다
> - [ ] Observer → scale 결정 → convert 박제의 흐름을 이해했다
> - [ ] int8 weight를 꺼내 복원 공식을 검증했다

---
# Part 3. ★ 캘리브레이션 대표성 실험 — Day 5 포팅 실패 #1을 미리 겪어보기

가상 NPU 실습 Part 6-4에서 numpy로 본 것을 **진짜 모델의 정확도**로 확인합니다.
교안 캘리브레이션 Best Practice의 세 조항을 하나씩 일부러 위반해 봅니다.

| 실험 조건 | 위반하는 조항 |
| --- | --- |
| A. 다양한 300장 (기준) | — 모범 사례 |
| B. 한 클래스(airplane)만 300장 | "클래스 균형 유지 — 편향 유발" |
| C. 딱 10장 | "충분한 샘플 수 — 최소 100장, 권장 300~500장" |
| D. CIFAR와 무관한 랜덤 노이즈 | "대표성 있는 데이터 — 실제 배포 환경 반영" |

### Step 3-1. 실험 파이프라인 함수화 — Part 2 전체를 함수 하나로

In [ ]:
def ptq_with_calib(base_model, calib_tensor, qconfig=None):
    """fuse → prepare → (주어진 데이터로) calibrate → convert 전체 파이프라인"""
    m = copy.deepcopy(base_model).eval()
    m.fuse_model()
    m.qconfig = qconfig or tq.get_default_qconfig("fbgemm")
    tq.prepare(m, inplace=True)
    with torch.no_grad():
        for i in range(0, len(calib_tensor), 64):
            m(calib_tensor[i:i+64])
    tq.convert(m, inplace=True)
    return m

print("파이프라인 함수 준비 완료 — 이제 캘리브레이션 데이터만 바꿔가며 실험합니다.")

### Step 3-2. 네 가지 캘리브레이션 데이터 준비

In [ ]:
# A. 다양한 300장 (모범)
calib_A = torch.stack([train_set[i][0] for i in range(300)])

# B. 한 클래스만 300장 — airplane(class 0)만 골라 모음
idx_airplane = [i for i in range(len(train_set)) if train_set[i][1] == 0][:300]
calib_B = torch.stack([train_set[i][0] for i in idx_airplane])

# C. 딱 10장
calib_C = calib_A[:10]

# D. 무관한 랜덤 노이즈 (분포 자체가 다름)
calib_D = torch.rand(300, 3, 32, 32) * 4 - 2

for name, c in [("A 다양 300장", calib_A), ("B 한클래스 300장", calib_B),
                ("C 10장", calib_C), ("D 노이즈 300장", calib_D)]:
    print(f"{name:16} shape={tuple(c.shape)}  값범위 [{c.min():.2f}, {c.max():.2f}]")

### Step 3-3. 실행 및 결과 비교 ★

In [ ]:
results = {}
for name, calib in [("A. 다양 300장 (모범)", calib_A), ("B. 한 클래스만", calib_B),
                    ("C. 딱 10장", calib_C), ("D. 랜덤 노이즈", calib_D)]:
    qm = ptq_with_calib(model, calib)
    results[name] = evaluate(qm, test_loader)

print(f"FP32 기준선: {fp32_acc:.2f}%\n")
print(f"{'캘리브레이션 조건':<22}{'INT8 정확도':>10}{'FP32 대비':>12}")
print("-" * 46)
for name, acc in results.items():
    print(f"{name:<24}{acc:>9.2f}%{acc - fp32_acc:>+10.2f}%p")

worst = min(results, key=results.get)
print(f"\n💥 최악: '{worst}' — 정확도 급락!")
print("💡 모델도, 코드도, 양자화 방법도 전부 같습니다. 오직 '어떤 데이터를 보여줬는가'만 달랐습니다.")
print("   Day 5 포팅 실패 #1 '정확도 급락 → 재캘리브레이션'의 원리를 미리 체험한 것입니다.")

> **✅ Part 3 확인**
> - [ ] 캘리브레이션 데이터의 다양성·수량·대표성이 각각 정확도에 미치는 영향을 실측했다
> - [ ] "재캘리브레이션"이 왜 포팅 실패의 1순위 처방인지 설명할 수 있다
>
> ✏️ **직접 해보기:** 조건 C의 장수를 10 → 30 → 100 → 300으로 늘리며 정확도 회복 곡선을 그려보세요.
> 교안의 "최소 100장" 기준이 여러분의 모델에서도 성립하나요?

---
# Part 4. Per-tensor vs Per-channel — 교안이 per-channel을 권장하는 이유

가상 NPU 실습에서는 per-tensor(텐서 전체에 scale 1개)만 구현했습니다.
실제 프레임워크로 두 방식을 정면 대결시키고, **왜 weight는 per-channel이 유리한지**
weight 분포를 직접 들여다보며 확인합니다.

### Step 4-1. 두 방식으로 각각 PTQ 실행

In [ ]:
# per-tensor: weight도 텐서 전체 scale 1개 (가상 NPU 실습 방식)
qconfig_pt = tq.QConfig(
    activation=tq.HistogramObserver.with_args(dtype=torch.quint8),
    weight=tq.MinMaxObserver.with_args(dtype=torch.qint8, qscheme=torch.per_tensor_symmetric))

# per-channel: 출력 채널마다 scale (fbgemm 기본값)
qconfig_pc = tq.get_default_qconfig("fbgemm")

qm_pt = ptq_with_calib(model, calib_A, qconfig_pt)
qm_pc = ptq_with_calib(model, calib_A, qconfig_pc)

acc_pt, acc_pc = evaluate(qm_pt, test_loader), evaluate(qm_pc, test_loader)
print(f"{'방식':<14}{'정확도':>9}{'FP32 대비':>11}")
print(f"{'per-tensor':<14}{acc_pt:>8.2f}%{acc_pt-fp32_acc:>+9.2f}%p")
print(f"{'per-channel':<14}{acc_pc:>8.2f}%{acc_pc-fp32_acc:>+9.2f}%p")
print(f"\n→ per-channel이 {acc_pc-acc_pt:+.2f}%p " +
      ("우세 — 교안 권장이 확인됩니다." if acc_pc >= acc_pt else
       "열세? 소형 모델·짧은 학습에선 차이가 작을 수 있습니다. 아래 분포로 원리를 확인하세요."))
print("💡 모델이 깊고 채널이 많을수록(ResNet-50, YOLO급) per-channel의 우위가 커집니다.")

### Step 4-2. 왜 그럴까 — 채널별 weight 범위를 눈으로 확인

per-tensor의 약점: **가장 큰 채널 하나가 scale을 독점**합니다.
범위가 작은 채널들은 INT8 256칸 중 몇 칸밖에 못 쓰게 되어 정밀도를 낭비합니다.

In [ ]:
ref = copy.deepcopy(model).eval(); ref.fuse_model()
w = ref.block2[0][0].weight.detach()                     # ConvReLU2d 내부 Conv2d, (64,32,3,3)
ch_max = w.abs().amax(dim=(1, 2, 3))                     # 채널별 |w| 최대

plt.figure(figsize=(10, 3.5))
plt.bar(range(len(ch_max)), ch_max.tolist(), alpha=0.8)
plt.axhline(ch_max.max().item(), ls="--", c="r",
            label=f"per-tensor scale 기준 = 전체 max ({ch_max.max():.3f})")
plt.xlabel("출력 채널 번호"); plt.ylabel("|weight| 최대값")
plt.title("block2 Conv — 채널마다 weight 범위가 제각각이다")
plt.legend(); plt.grid(axis="y", alpha=0.3); plt.tight_layout(); plt.show()

ratio = (ch_max.max() / ch_max.min()).item()
small_ch = (ch_max < ch_max.max() / 4).sum().item()
levels = int(255 / ratio)
print(f"최대/최소 채널 범위 비율: {ratio:.1f}배")
print(f"per-tensor라면: 범위가 가장 작은 채널은 INT8 256칸 중 약 {levels}칸만 사용 — 정밀도 낭비!")
print(f"전체 {len(ch_max)}채널 중 {small_ch}개가 최대치의 1/4 미만 범위 → per-channel로 각자 scale을 갖는 게 이득")

> **✅ Part 4 확인**
> - [ ] per-tensor는 최대 채널이 scale을 독점해 작은 채널의 정밀도를 낭비함을 시각적으로 확인했다
> - [ ] "weight는 per-channel, activation은 per-tensor"가 fbgemm 기본값인 이유를 설명할 수 있다
>   (activation은 배치마다 바뀌어 채널별 통계가 불안정 + HW 구현 복잡)
> - [ ] 가상 NPU 실습 6장의 "간헐적 NaN/Inf → per-channel로 완화"가 같은 원리임을 연결했다

---
# Part 5. 추론 속도 실측 — INT8은 CPU에서도 빨라진다

Day 5 벤치마크 표준(warmup 제외, 반복 평균)대로 FP32와 INT8의 CPU 추론 속도를 잽니다.

In [ ]:
fp32_bench = bench_ms(model)
int8_bench = bench_ms(qm_pc)

print(f"{'':14}{'FP32':>10}{'INT8':>10}")
print(f"{'지연(ms)':12}{fp32_bench:>10.1f}{int8_bench:>10.1f}")
print(f"{'속도 향상':11}{'':>10}{fp32_bench/int8_bench:>9.1f}x")
print()
print("💡 CPU에서도 1.5~3배가 일반적입니다 (정수 SIMD + 메모리 대역폭 4배 절감).")
print("   전용 NPU에서는 여기에 '같은 면적에 8배 많은 MAC'(가상실습 6.1절)이 곱해져")
print("   교안의 'ResNet50 330ms → 33ms(10배)'같은 도약이 나옵니다.")
print("\n✏️ 직접 해보기: batch=1로 재측정해 보세요. 속도 이득이 줄어드나요?")
print("   (교안 7.2절 'batch=1 실측은 훨씬 낮음'을 CPU에서도 관찰할 수 있습니다)")

---
# Part 6. (선택) QAT 맛보기 — PTQ→QAT 결정 규칙 체험

교안 결정 규칙: "*먼저 PTQ 시도 → 정확도 손실 > 2%p이면 QAT로 전환*".
Part 2에서 손실이 2%p 이내였다면 QAT는 불필요하지만, **전환하는 법 자체**를 여기서 익혀 둡니다.
Day 2 본 실습의 2단계 학습(`hyp.optimize.yaml`)이 바로 이 QAT의 디퍼아이 버전입니다.

### Step 6-1. QAT 준비 — 순서가 생명

```text
eval()에서 fuse  →  train() 전환  →  qat_qconfig  →  prepare_qat (FakeQuantize 삽입)
```

⚠️ **흔한 실수:** train 모드에서 fuse하면 `AssertionError: Fusion only for eval!` — 순서를 지키세요.

In [ ]:
qat_model = copy.deepcopy(model)
qat_model.eval(); qat_model.fuse_model()          # ① 반드시 eval에서 fuse
qat_model.train()                                  # ② train 모드 전환
qat_model.qconfig = tq.get_default_qat_qconfig("fbgemm")
tq.prepare_qat(qat_model, inplace=True)            # ③ FakeQuantize 삽입

print(type(qat_model.block1[0]).__name__, "— weight_fake_quant가 포함된 QAT 모듈")
print("\n💡 FakeQuantize = 순전파에서 양자화를 '흉내'내되(round+clip) 역전파는 통과(STE).")
print("   모델이 학습 중에 양자화 오차를 미리 겪고 스스로 보정하게 만드는 장치입니다.")

### Step 6-2. 짧은 QAT 재학습 — 교안 하이퍼파라미터 가이드 적용

교안 QAT 가이드: "*Epoch은 원본의 10~20%, LR은 원본의 1/100*".
원본 학습이 3 epoch·lr 0.05였으니 → **1 epoch·lr 0.0005**로 재학습합니다.
(CPU 모드에서는 배치 수를 제한해 시간을 아낍니다)

In [ ]:
qat_model = qat_model.to(device)
opt = torch.optim.SGD(qat_model.parameters(), lr=0.0005, momentum=0.9)   # 원본 lr의 1/100
max_batches = None if device == "cuda" else 40                            # CPU면 40배치만

qat_model.train(); t0 = time.time()
for bi, (x, y) in enumerate(train_loader):
    if max_batches and bi >= max_batches: break
    x, y = x.to(device), y.to(device)
    opt.zero_grad(); loss = crit(qat_model(x), y); loss.backward(); opt.step()
print(f"QAT 재학습 완료 ({time.time()-t0:.0f}s, 마지막 loss {loss.item():.3f})")

qat_model = qat_model.cpu().eval()
qat_int8 = tq.convert(copy.deepcopy(qat_model), inplace=False)   # ④ INT8 변환
qat_acc = evaluate(qat_int8, test_loader)

### Step 6-3. 최종 대결 — FP32 vs PTQ vs QAT

In [ ]:
print(f"{'모델':<16}{'정확도':>9}{'FP32 대비':>11}{'크기':>9}")
print("-" * 46)
print(f"{'FP32 원본':<17}{fp32_acc:>8.2f}%{'—':>10}{fp32_size:>7.0f}KB")
print(f"{'PTQ (per-ch)':<16}{acc_pc:>8.2f}%{acc_pc-fp32_acc:>+9.2f}%p{model_size_kb(qm_pc):>7.0f}KB")
print(f"{'QAT':<16}{qat_acc:>8.2f}%{qat_acc-fp32_acc:>+9.2f}%p{model_size_kb(qat_int8):>7.0f}KB")
print()
if qat_acc >= acc_pc:
    print(f"→ QAT가 PTQ 대비 {qat_acc-acc_pc:+.2f}%p 회복 — 교안 'QAT 손실 0.5%p 이내'로 가는 방향입니다.")
else:
    print("→ 이번엔 QAT 이득이 안 보이나요? 재학습이 너무 짧았을 수 있습니다 (특히 CPU 축소 모드).")
    print("  Detection처럼 민감한 태스크·깊은 모델일수록 QAT의 가치가 커집니다 (교안 'QAT를 써야 하는 경우').")
print("\n💡 Day 2 본 실습의 2단계 학습(bsnet-t → bsnet-t-o)이 정확히 이 구조입니다:")
print("   1단계 = FP32 학습(우리의 Part 1) / 2단계 = 최적화 인지 재학습(우리의 QAT) — 도구만 DDesignerAPI로.")

---
# Part 7. 리포트 과제 & Day 2 연결

## 제출 과제 — `quant_report.md`

**A. 결과표 채우기** (본인 런타임 기준)

| 항목 | FP32 | PTQ per-tensor | PTQ per-channel | QAT |
| --- | --- | --- | --- | --- |
| 정확도 (%) | | | | |
| FP32 대비 (%p) | — | | | |
| 크기 (KB) | | | | |
| CPU 지연 (ms) | | — | | — |

**B. 캘리브레이션 실험표** (Part 3 결과)

| 조건 | 정확도 | 하락폭 |
| --- | --- | --- |
| A 다양 300장 | | |
| B 한 클래스 | | |
| C 10장 | | |
| D 노이즈 | | |

**C. 분석 질문 (각 2~3문장)**
1. 교안 결정 규칙(PTQ 손실 2%p 기준)을 여러분의 결과에 적용하면 결론은 PTQ인가 QAT인가?
2. 조건 B(한 클래스)가 조건 C(10장)보다 나쁜/좋은 이유를 "분포 관찰" 관점에서 설명하시오.
3. Day 2 본 실습에서 캘리브레이션 데이터로 무엇을 쓰는 것이 이상적일까? (힌트: Day 1의 MIPI 카메라)

**D. 스크린샷**: Part 2-8의 int8 weight 해부 출력, Part 3 결과표, Part 4 채널 분포 그래프

## 오늘 배운 것 ↔ 다음 단계 대응표

| 이 실습 (PyTorch) | Day 2 본 실습 (디퍼아이) | 가상 NPU 실습 (원리) |
| --- | --- | --- |
| `fuse_modules` | 컴파일러의 그래프 최적화(fusion) | Part 10 fusion 개념 |
| Observer + calibration | 캘리브레이션 데이터 주입 | Part 6 `calib_max` |
| `convert` (scale 박제) | `.tachyrt`에 상수 박제 | Part 6 재양자화 상수 |
| `prepare_qat` + 재학습 | 2단계 최적화 학습 (`hyp.optimize.yaml`, XWN) | — |
| fbgemm (x86 CPU 백엔드) | TACHY-Runtime (NPU 네이티브) | `VirtualNPU` 클래스 |

## ✏️ 심화 도전 과제 (선택)

1. **캘리브레이션 곡선**: 장수를 [10, 30, 100, 300, 1000]으로 바꿔가며 정확도 곡선을 그리고 "충분한 장수"의 변곡점을 찾기
2. **Observer 비교**: MinMaxObserver vs HistogramObserver vs MovingAverageMinMaxObserver로 각각 PTQ 후 정확도 비교 (교안 "MinMax·Percentile·KL-Divergence 범위 전략"의 축소판)
3. **민감 레이어 찾기**: 한 블록씩만 양자화에서 제외(qconfig=None)하며 정확도 변화 측정 → 어느 레이어가 양자화에 가장 민감한가?
4. **ONNX Export 연결**: `qat_int8`을 두고, FP32 원본 모델을 opset 13·고정 shape으로 Export한 뒤 Netron으로 열어보기 (2순위 실습 예고편)

---
수고하셨습니다! 🎉 이제 여러분은 양자화를 **원리(numpy) → 도구(PyTorch)** 두 층위로 다룰 수 있습니다.
Day 2에서 세 번째 층위 — **실전(디퍼아이 툴체인 → 실물 NPU)** — 를 완성하게 됩니다.
